In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

In [2]:
# df = pd.read_csv(r'C:\Users\nickc\OneDrive\Documents\Projects\Sports-Analytics\Data\MLBAM17.csv')
df = pd.read_csv(r'C:\Users\nick.chedzoy_predict\Documents\Python\Sports-Analytics\Module2_Moneyball_and_Beyond\Data\Week 4\MLBAM17.csv')

In [3]:
df.drop(columns=list(df.columns)[0], inplace=True)

In [4]:
# Columns needed for Run Expectancy Matrix:

mtrx_cols = [
    'batterName','batterId','event', 'start1B', 'start2B', 'start3B', 'end1B', 'end2B', 'end3B',
    'startOuts','endOuts','runsFuture','runsOnPlay','outsInInning', 'stand', 'throws','venueId', 
    'stadium', 'batterPos']

RE17 = df[mtrx_cols].copy()

In [5]:
# Tag if player on base = True

RE17['Start1'] = RE17['start1B'].notna().astype(int)
RE17['Start2'] = RE17['start2B'].notna().astype(int)
RE17['Start3'] = RE17['start3B'].notna().astype(int)

In [6]:
# Create 'Starting State'

RE17['start_state'] = (
    RE17['Start1'].astype(str) + RE17['Start2'].astype(str) + RE17['Start3'].astype(str) +
    " " + RE17['startOuts'].astype(str)
    )

In [7]:
# Tasks 1 and 2: ending base occupancy (1 = occupied, 0 = empty) and base-out state

RE17['End1'] = RE17['end1B'].notna().astype(int)
RE17['End2'] = RE17['end2B'].notna().astype(int)
RE17['End3'] = RE17['end3B'].notna().astype(int)

RE17['End_state'] = (
    RE17['End1'].astype(str) + RE17['End2'].astype(str) + RE17['End3'].astype(str) +
    " " + RE17['endOuts'].astype(str)
    )

In [8]:
# Task 3: keep state-changing or run-scoring plays in innings with exactly 3 outs

RE17 = RE17.loc[
    ((RE17['start_state'] != RE17['End_state']) | (RE17['runsOnPlay'] > 0))
    & (RE17['outsInInning'] == 3)
].copy()

In [9]:
# Task 4: average runs from the start of each play to the end of the inning
# runsFuture includes any runs scored on the current play.

Start_RunExp = (
    RE17.groupby('start_state', as_index=False)['runsFuture']
    .mean()
    .rename(columns={'runsFuture': 'Start_RE'})
)

display(Start_RunExp)

,start_state,Start_RE
0,000 0,0.516375
1,000 1,0.272176
2,000 2,0.108038
3,001 0,1.436482
4,001 1,0.953586
5,001 2,0.381186
6,010 0,1.154829
7,010 1,0.711650
8,010 2,0.336702
9,011 0,2.051376


In [10]:
# Merge starting-state expectancy onto each retained play

RE17 = RE17.merge(
    Start_RunExp, on='start_state', how='left', validate='many_to_one'
)

In [11]:
# Task 5: use the same expectancy for ending states with 0, 1 or 2 outs
# All eight base configurations with 3 outs have zero future runs.

Base_State_3 = pd.DataFrame({
    'start_state': [f'{bases:03b} 3' for bases in range(8)],
    'Start_RE': 0.0
})

End_RunExp = pd.concat([Start_RunExp, Base_State_3], ignore_index=True).rename(
    columns={'start_state': 'End_state', 'Start_RE': 'End_RE'}
)

display(End_RunExp)

,End_state,End_RE
0,000 0,0.516375
1,000 1,0.272176
2,000 2,0.108038
3,001 0,1.436482
4,001 1,0.953586
5,001 2,0.381186
6,010 0,1.154829
7,010 1,0.711650
8,010 2,0.336702
9,011 0,2.051376


In [12]:
# Task 6: merge ending-state expectancy onto each retained play

RE17 = RE17.merge(
    End_RunExp, on='End_state', how='left', validate='many_to_one'
)

In [13]:
# Task 7: run value = runs scored + ending expectancy - starting expectancy

RE17['Run_Value'] = RE17['runsOnPlay'] + RE17['End_RE'] - RE17['Start_RE']

display(RE17[[
    'batterName', 'event', 'start_state', 'End_state',
    'runsOnPlay', 'Start_RE', 'End_RE', 'Run_Value'
]].head(10))

,batterName,event,start_state,End_state,runsOnPlay,Start_RE,End_RE,Run_Value
0,Gardner,Flyout,000 0,000 1,0,0.516375,0.272176,-0.244199
1,"Sanchez, G",Groundout,000 1,000 2,0,0.272176,0.108038,-0.164138
2,Bird,Walk,000 2,100 2,0,0.108038,0.225365,0.117327
3,Holliday,Groundout,100 2,000 3,0,0.225365,0.000000,-0.225365
4,"Dickerson, C",Single,000 0,100 0,0,0.516375,0.912921,0.396546
5,Kiermaier,Double,100 0,011 0,0,0.912921,2.051376,1.138455
6,Longoria,Sac Fly,011 0,010 1,1,2.051376,0.711650,-0.339727
7,"Miller, B",Single,010 1,101 1,0,0.711650,1.215306,0.503657
8,Souza Jr.,Walk,101 1,111 1,0,1.215306,1.635608,0.420302
9,Morrison,Single,111 1,110 1,2,1.635608,0.947424,1.311816


The following summaries use the filtered `RE17` data. Bases loaded means all three starting-base indicators equal 1, regardless of the number of outs. For strikeouts, `Strikeouts` counts events labelled `Strikeout`; `Total_Strikeouts` also includes `Strikeout - DP`.

In [14]:
# Count plate appearances starting with the bases loaded

Bases_Loaded_PA = int(RE17[['Start1', 'Start2', 'Start3']].eq(1).all(axis=1).sum())
print(f'Plate appearances starting with bases loaded: {Bases_Loaded_PA:,}')

Plate appearances starting with bases loaded: 4,364


In [15]:
# Sum strikeout indicators directly by player position

Strikeouts_by_Position = (
    RE17.assign(
        Strikeouts=RE17['event'].eq('Strikeout').astype(int),
        Strikeout_DP=RE17['event'].eq('Strikeout - DP').astype(int)
    )
    .groupby('batterPos', as_index=False, dropna=False)[['Strikeouts', 'Strikeout_DP']]
    .sum()
)
Strikeouts_by_Position['Total_Strikeouts'] = (
    Strikeouts_by_Position['Strikeouts'] + Strikeouts_by_Position['Strikeout_DP']
)
Strikeouts_by_Position = Strikeouts_by_Position.sort_values(
    'Total_Strikeouts', ascending=False, ignore_index=True
)

display(Strikeouts_by_Position)

Highest_Strikeouts = Strikeouts_by_Position.loc[
    Strikeouts_by_Position['Strikeouts'].idxmax()
]
Highest_Total_Strikeouts = Strikeouts_by_Position.iloc[0]
print(
    f"Highest aggregate Strikeout count: {Highest_Strikeouts['batterPos']} "
    f"({Highest_Strikeouts['Strikeouts']:,})"
)
print(
    f"Highest total including Strikeout - DP: {Highest_Total_Strikeouts['batterPos']} "
    f"({Highest_Total_Strikeouts['Total_Strikeouts']:,})"
)

,batterPos,Strikeouts,Strikeout_DP,Total_Strikeouts
0,1B,4399,21,4420
1,RF,4386,22,4408
2,CF,4374,14,4388
3,LF,4317,16,4333
4,C,4090,16,4106
5,3B,4023,22,4045
6,SS,3734,12,3746
7,2B,3710,11,3721
8,UN,2594,6,2600
9,DH,2354,6,2360


Highest aggregate Strikeout count: 1B (4,399)
Highest total including Strikeout - DP: 1B (4,420)


In [16]:
MLBAM16 = pd.read_csv(r'Data\Week 4\MLBAM16.csv')

In [17]:
# Part II (2016), step 2: keep the same relevant columns as for 2017
# Selecting mtrx_cols also excludes the CSV's exported index column.

RE16 = MLBAM16[mtrx_cols].copy()

In [18]:
# Step 3: starting base occupancy (1 = occupied, 0 = empty)

RE16['Start1'] = RE16['start1B'].notna().astype(int)
RE16['Start2'] = RE16['start2B'].notna().astype(int)
RE16['Start3'] = RE16['start3B'].notna().astype(int)

In [19]:
# Step 4: combine starting base occupancy and outs into a base-out state

RE16['start_state'] = (
    RE16['Start1'].astype(str) + RE16['Start2'].astype(str) + RE16['Start3'].astype(str)
    + ' ' + RE16['startOuts'].astype(str)
)

In [20]:
# Step 5: ending base occupancy (1 = occupied, 0 = empty)

RE16['End1'] = RE16['end1B'].notna().astype(int)
RE16['End2'] = RE16['end2B'].notna().astype(int)
RE16['End3'] = RE16['end3B'].notna().astype(int)

In [21]:
# Step 6: combine ending base occupancy and outs into a base-out state

RE16['End_state'] = (
    RE16['End1'].astype(str) + RE16['End2'].astype(str) + RE16['End3'].astype(str)
    + ' ' + RE16['endOuts'].astype(str)
)

In [22]:
# Step 7: keep state-changing or run-scoring plays in innings with exactly 3 outs

RE16 = RE16.loc[
    ((RE16['start_state'] != RE16['End_state']) | (RE16['runsOnPlay'] > 0))
    & (RE16['outsInInning'] == 3)
].copy()

print(f'2016 plays retained: {len(RE16):,} of {len(MLBAM16):,}')

2016 plays retained: 184,152 of 184,935


In [23]:
# Step 8: calculate starting-state expectancy from the 2016 data only
# runsFuture includes runs scored on the current play.

Start_RunExp16 = (
    RE16.groupby('start_state', as_index=False)['runsFuture']
    .mean()
    .rename(columns={'runsFuture': 'Start_RE'})
)

display(Start_RunExp16)

,start_state,Start_RE
0,000 0,0.498377
1,000 1,0.268678
2,000 2,0.106305
3,001 0,1.358166
4,001 1,0.948684
5,001 2,0.377197
6,010 0,1.132468
7,010 1,0.677498
8,010 2,0.319844
9,011 0,1.916190


In [24]:
# Merge 2016 starting-state expectancy onto each retained 2016 play

RE16 = RE16.merge(
    Start_RunExp16, on='start_state', how='left', validate='many_to_one'
)

In [25]:
# Step 9: build the 2016 ending-state lookup from starting-state expectancy
# All eight base configurations with 3 outs have zero future runs.

Base_State_3_16 = pd.DataFrame({
    'start_state': [f'{bases:03b} 3' for bases in range(8)],
    'Start_RE': 0.0
})

End_RunExp16 = pd.concat([Start_RunExp16, Base_State_3_16], ignore_index=True).rename(
    columns={'start_state': 'End_state', 'Start_RE': 'End_RE'}
)

display(End_RunExp16)

,End_state,End_RE
0,000 0,0.498377
1,000 1,0.268678
2,000 2,0.106305
3,001 0,1.358166
4,001 1,0.948684
5,001 2,0.377197
6,010 0,1.132468
7,010 1,0.677498
8,010 2,0.319844
9,011 0,1.916190


In [26]:
# Step 10: merge 2016 ending-state expectancy onto each retained 2016 play

RE16 = RE16.merge(
    End_RunExp16, on='End_state', how='left', validate='many_to_one'
)

In [27]:
# Step 11: run value = runs scored + ending expectancy - starting expectancy

RE16['Run_Value'] = RE16['runsOnPlay'] + RE16['End_RE'] - RE16['Start_RE']

display(RE16[[
    'batterName', 'event', 'start_state', 'End_state',
    'runsOnPlay', 'Start_RE', 'End_RE', 'Run_Value'
]].head(10))

,batterName,event,start_state,End_state,runsOnPlay,Start_RE,End_RE,Run_Value
0,"Carpenter, M",Groundout,000 0,000 1,0,0.498377,0.268678,-0.229699
1,Pham,Groundout,000 1,000 2,0,0.268678,0.106305,-0.162373
2,Holliday,Strikeout,000 2,000 3,0,0.106305,0.000000,-0.106305
3,Jaso,Groundout,000 0,000 1,0,0.498377,0.268678,-0.229699
4,McCutchen,Hit By Pitch,000 1,100 1,0,0.268678,0.512225,0.243547
5,Freese,Single,100 1,101 1,0,0.512225,1.196777,0.684552
6,"Marte, S",Lineout,101 1,101 2,0,1.196777,0.480175,-0.716603
7,Cervelli,Pop Out,101 2,000 3,0,0.480175,0.000000,-0.480175
8,Grichuk,Strikeout,000 0,000 1,0,0.498377,0.268678,-0.229699
9,Piscotty,Strikeout,000 1,000 2,0,0.268678,0.106305,-0.162373


These 2016 summaries use the filtered `RE16` data. Ground outs are events labelled `Groundout`. An occupied starting base means at least one of `Start1`, `Start2`, or `Start3` equals 1. Home-run indicators are summed directly by `batterPos`.

In [28]:
# Percentage of 2016 plate appearances resulting in ground outs

Groundout_Count16 = int(RE16['event'].eq('Groundout').sum())
Groundout_Percent16 = 100 * Groundout_Count16 / len(RE16)
print(
    f'Ground outs: {Groundout_Count16:,} / {len(RE16):,} '
    f'= {Groundout_Percent16:.2f}%'
)

Ground outs: 33,967 / 184,152 = 18.45%


In [29]:
# Count 2016 plate appearances starting with at least one occupied base

Occupied_Base_PA16 = int(RE16[['Start1', 'Start2', 'Start3']].eq(1).any(axis=1).sum())
print(f'Plate appearances starting with at least one occupied base: {Occupied_Base_PA16:,}')

Plate appearances starting with at least one occupied base: 80,387


In [30]:
# Sum 2016 home-run indicators by player position

Home_Runs_by_Position16 = (
    RE16.assign(Home_Runs=RE16['event'].eq('Home Run').astype(int))
    .groupby('batterPos', as_index=False, dropna=False)['Home_Runs']
    .sum()
    .sort_values('Home_Runs', ascending=False, ignore_index=True)
)

display(Home_Runs_by_Position16)
Highest_Home_Runs16 = Home_Runs_by_Position16.iloc[0]
print(
    f"Highest aggregate home run total: {Highest_Home_Runs16['batterPos']} "
    f"({Highest_Home_Runs16['Home_Runs']:,})"
)

,batterPos,Home_Runs
0,1B,797
1,3B,710
2,RF,694
3,2B,576
4,LF,576
5,C,541
6,CF,532
7,SS,486
8,DH,424
9,UN,178


Highest aggregate home run total: 1B (797)


Compare run values across 2016 and 2017 using the filtered `RE16` and `RE17` data. Event values are means; player values are sums. The event comparison retains all event types from either season, with missing values for any event absent in a season. Players are matched by `batterId`, retaining only players present in both seasons and displaying their 2017 name where available.

In [31]:
# Task 1: average run value of each event in each season

Event_Value16 = (
    RE16.groupby('event', as_index=False)['Run_Value']
    .mean()
    .rename(columns={'Run_Value': 'Run_Value16'})
)
Event_Value17 = (
    RE17.groupby('event', as_index=False)['Run_Value']
    .mean()
    .rename(columns={'Run_Value': 'Run_Value17'})
)

display(Event_Value16)
display(Event_Value17)

,event,Run_Value16
0,Batter Interference,-0.284649
1,Bunt Groundout,-0.218826
2,Bunt Lineout,-0.352295
3,Bunt Pop Out,-0.342802
4,Catcher Interference,0.301623
5,Double,0.743467
6,Double Play,-0.864981
7,Fan interference,0.533316
8,Field Error,0.469989
9,Fielders Choice,0.701447


,event,Run_Value17
0,Batter Interference,-0.430019
1,Bunt Groundout,-0.209411
2,Bunt Lineout,-0.328292
3,Bunt Pop Out,-0.373225
4,Catcher Interference,0.399070
5,Double,0.779338
6,Double Play,-0.897164
7,Fan interference,0.590743
8,Field Error,0.493206
9,Fielders Choice,0.764112


In [32]:
# Task 2: merge the seasonal event averages by event

Event_Run_Values = Event_Value16.merge(
    Event_Value17, on='event', how='outer', validate='one_to_one'
)

display(Event_Run_Values)

,event,Run_Value16,Run_Value17
0,Batter Interference,-0.284649,-0.430019
1,Bunt Groundout,-0.218826,-0.209411
2,Bunt Lineout,-0.352295,-0.328292
3,Bunt Pop Out,-0.342802,-0.373225
4,Catcher Interference,0.301623,0.399070
5,Double,0.743467,0.779338
6,Double Play,-0.864981,-0.897164
7,Fan interference,0.533316,0.590743
8,Field Error,0.469989,0.493206
9,Fielders Choice,0.701447,0.764112


In [33]:
# Task 3: sum run values for each player in each season
# Use player IDs so different players with the same name remain separate.

Player_Value16 = RE16.groupby('batterId', as_index=False).agg(
    batterName=('batterName', 'first'),
    Run_Value16=('Run_Value', 'sum')
)
Player_Value17 = RE17.groupby('batterId', as_index=False).agg(
    batterName=('batterName', 'first'),
    Run_Value17=('Run_Value', 'sum')
)

print(f'2016 players: {len(Player_Value16):,}')
display(Player_Value16.head(10))
print(f'2017 players: {len(Player_Value17):,}')
display(Player_Value17.head(10))

2016 players: 968


,batterId,batterName,Run_Value16
0,112526,Colon,-15.380841
1,120074,Ortiz,60.938730
2,121347,"Rodriguez, A",-10.691410
3,134181,Beltre,29.745115
4,136860,Beltran,21.279393
5,150029,Werth,6.368448
6,150229,Pierzynski,-17.583062
7,276519,Rollins,-7.574579
8,279571,Belisle,-0.481289
9,282332,Sabathia,-0.944580


2017 players: 957


,batterId,batterName,Run_Value17
0,112526,Colon,-6.053060
1,134181,Beltre,27.205050
2,136860,Beltran,-16.883042
3,150029,Werth,-7.246145
4,276520,"Arroyo, B",-3.530974
5,282332,Sabathia,-0.292495
6,285079,Dickey,-11.955467
7,400085,"Suzuki, I",1.083044
8,400121,"Martinez, V",-16.021087
9,400284,Utley,-1.799883


In [34]:
# Task 4: keep only players with recorded plays in both seasons

Player_Run_Values = Player_Value16.merge(
    Player_Value17, on='batterId', how='inner',
    suffixes=('_16', '_17'), validate='one_to_one'
)
Player_Run_Values['batterName'] = Player_Run_Values['batterName_17'].combine_first(
    Player_Run_Values['batterName_16']
)
Player_Run_Values = Player_Run_Values[[
    'batterId', 'batterName', 'Run_Value16', 'Run_Value17'
]]

print(f'Players appearing in both seasons: {len(Player_Run_Values):,}')
display(Player_Run_Values.head(10))

Players appearing in both seasons: 691


,batterId,batterName,Run_Value16,Run_Value17
0,112526,Colon,-15.380841,-6.053060
1,134181,Beltre,29.745115,27.205050
2,136860,Beltran,21.279393,-16.883042
3,150029,Werth,6.368448,-7.246145
4,282332,Sabathia,-0.944580,-0.292495
5,285079,Dickey,-0.291686,-11.955467
6,400085,"Suzuki, I",-2.577561,1.083044
7,400121,"Martinez, V",7.335464,-16.021087
8,400284,Utley,6.689794,-1.799883
9,405395,Pujols,15.581010,-13.892055


Seasonal player extremes use all players in each season. Changes are calculated as 2017 minus 2016. The largest event change is ranked by absolute difference, using events with values in both seasons. Player improvement is ranked by signed difference among players present in both seasons.

In [35]:
# Highest player run value in 2017 and lowest player run value in 2016

Highest_Player_Value17 = Player_Value17.loc[Player_Value17['Run_Value17'].idxmax()]
Lowest_Player_Value16 = Player_Value16.loc[Player_Value16['Run_Value16'].idxmin()]
print(
    f"Highest player run value in 2017: {Highest_Player_Value17['batterName']} "
    f"({Highest_Player_Value17['Run_Value17']:.6f})"
)
print(
    f"Lowest player run value in 2016: {Lowest_Player_Value16['batterName']} "
    f"({Lowest_Player_Value16['Run_Value16']:.6f})"
)

Highest player run value in 2017: Votto (67.650742)
Lowest player run value in 2016: Hechavarria (-32.142529)


In [36]:
# Event run value changes: 2017 minus 2016

Event_Run_Values['Run_Value_Difference'] = (
    Event_Run_Values['Run_Value17'] - Event_Run_Values['Run_Value16']
)
Event_Run_Values['Absolute_Run_Value_Difference'] = (
    Event_Run_Values['Run_Value_Difference'].abs()
)
display(Event_Run_Values.sort_values('Absolute_Run_Value_Difference', ascending=False))

Largest_Event_Change = Event_Run_Values.loc[
    Event_Run_Values['Absolute_Run_Value_Difference'].idxmax()
]
print(
    f"Largest absolute event change: {Largest_Event_Change['event']}; "
    f"2016 = {Largest_Event_Change['Run_Value16']:.6f}, "
    f"2017 = {Largest_Event_Change['Run_Value17']:.6f}, "
    f"difference = {Largest_Event_Change['Run_Value_Difference']:+.6f}"
)

,event,Run_Value16,Run_Value17,Run_Value_Difference,Absolute_Run_Value_Difference
23,Sac Fly DP,-0.370443,-0.543715,-0.173272,0.173272
29,Triple Play,-1.637543,-1.471223,0.166319,0.166319
0,Batter Interference,-0.284649,-0.430019,-0.145369,0.145369
4,Catcher Interference,0.301623,0.399070,0.097447,0.097447
9,Fielders Choice,0.701447,0.764112,0.062665,0.062665
27,Strikeout - DP,-0.622787,-0.682819,-0.060032,0.060032
7,Fan interference,0.533316,0.590743,0.057427,0.057427
22,Sac Fly,0.023233,-0.032119,-0.055352,0.055352
10,Fielders Choice Out,-0.610757,-0.662400,-0.051643,0.051643
13,Grounded Into DP,-0.773318,-0.818005,-0.044687,0.044687


Largest absolute event change: Sac Fly DP; 2016 = -0.370443, 2017 = -0.543715, difference = -0.173272


In [37]:
# Player run value changes for players appearing in both seasons

Player_Run_Values['Run_Value_Difference'] = (
    Player_Run_Values['Run_Value17'] - Player_Run_Values['Run_Value16']
)
display(Player_Run_Values.sort_values('Run_Value_Difference', ascending=False).head(10))

Largest_Player_Improvement = Player_Run_Values.loc[
    Player_Run_Values['Run_Value_Difference'].idxmax()
]
print(
    f"Largest player improvement: {Largest_Player_Improvement['batterName']}; "
    f"2016 = {Largest_Player_Improvement['Run_Value16']:.6f}, "
    f"2017 = {Largest_Player_Improvement['Run_Value17']:.6f}, "
    f"improvement = {Largest_Player_Improvement['Run_Value_Difference']:+.6f}"
)

,batterId,batterName,Run_Value16,Run_Value17,Run_Value_Difference
501,592450,Judge,-4.179157,57.081625,61.260782
198,475582,Zimmerman,-22.942623,33.179676,56.122300
260,503556,"Gonzalez, M",-16.542950,37.777259,54.320209
314,519317,Stanton,10.070038,51.784088,41.714049
663,624424,Conforto,-3.607318,35.827160,39.434477
336,542303,Ozuna,7.040069,45.648106,38.608037
437,571697,Gennett,-8.970409,29.352662,38.323071
76,446359,Cozart,-2.402277,33.761590,36.163867
196,475253,Smoak,-12.844614,22.263561,35.108175
331,541645,"Garcia, Av",0.792947,34.612266,33.819319


Largest player improvement: Judge; 2016 = -4.179157, 2017 = 57.081625, improvement = +61.260782
